# Fine-Tuning Laya on the GHAST AIS spoofing dataset (Kaggle GPU T4 x2, DDP)

Adapted from Laya's `laya_finetune_typed_decisions_2xT4_kaggle.ipynb`. The training method is unchanged
(RLCD policy gradient + soft cross-entropy, GRPO-style baseline, held-out calibration slice, temperature fit,
`torchrun --nproc_per_node=2` DDP, same hyperparameters).

**Pipeline:** GitHub `Vishmayraj/GHAST` -> `data/laya/{train,holdout}.jsonl` -> fine-tune `convaiinnovations/laya` -> evaluate on the 1,500-row holdout -> export checkpoint for `ml/laya_model/`.

Nothing is pushed to Hugging Face.

## Kaggle setup (do this before "Run All")

In the right-hand sidebar -> **Notebook options** (Settings):

| Setting | Value |
|---|---|
| **Accelerator** | **GPU T4 x2** (the DDP script needs both GPUs) |
| **Internet** | **On** (git clone + Hugging Face model download + pip) |
| **Persistence** | Optional. "Files only" keeps `/kaggle/working` between sessions |
| **Secrets / HF token** | Not needed (`convaiinnovations/laya` is public) |

Then **Save Version -> Save & Run All (Commit)** if you want the checkpoint to appear in the notebook's **Output** tab. Interactive runs are lost when the session ends.
Checkpoints and reports are written to `/kaggle/working`.

**GPU quota:** an interactive session keeps billing GPU time until you stop it, even when every cell has finished. Prefer **Save & Run All (Commit)**, which releases the GPU when the run ends. If you ran interactively, stop the session from the sessions panel (bottom-left "Sessions" / power icon) when done.

## 1. Environment & Dual T4 GPU check

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Switch the Kaggle Accelerator: right sidebar -> Notebook options -> Accelerator -> GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")

## 2. Install dependencies

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("Datasets version    :", datasets.__version__)
print("PyTorch version     :", torch.__version__)

## 3. Get the GHAST repo and dataset
Clones the public GitHub repo into `/kaggle/working/GHAST` and checks that `data/laya/{train.jsonl,holdout.jsonl,manifest.json}` are present.
(Fallback: if you attached the files as a Kaggle Dataset instead, the search below also looks under `/kaggle/input`.)

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Vishmayraj/GHAST.git"
REPO_BRANCH = "main"                       # change if the dataset lives on another branch
WORK_DIR = Path("/kaggle/working")
CLONE_DIR = WORK_DIR / "GHAST"

if not (CLONE_DIR / "data" / "laya" / "train.jsonl").exists():
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)], check=True)

def _find_data_dir():
    cands = [CLONE_DIR / "data" / "laya"]
    for root in Path("/kaggle/input").glob("*"):          # optional: files attached as a Kaggle Dataset
        cands += [root / "data" / "laya", root / "laya", root]
    for c in cands:
        if (c / "train.jsonl").exists() and (c / "holdout.jsonl").exists():
            return c
    raise FileNotFoundError("train.jsonl / holdout.jsonl not found. Check Internet is On and REPO_URL/REPO_BRANCH.")

DATA_DIR = _find_data_dir()
TRAIN_JSONL = DATA_DIR / "train.jsonl"
HOLDOUT_JSONL = DATA_DIR / "holdout.jsonl"
REPO_ROOT = CLONE_DIR

ITEMS_PATH = WORK_DIR / "train_items.pt"
SCRIPT_PATH = WORK_DIR / "train_ddp.py"
OUTPUT_DIR = WORK_DIR / "laya_finetuned_ghast"

print("Data dir:", DATA_DIR)
print("train   :", sum(1 for _ in open(TRAIN_JSONL, encoding="utf-8")), "rows")
print("holdout :", sum(1 for _ in open(HOLDOUT_JSONL, encoding="utf-8")), "rows")
man = json.load(open(DATA_DIR / "manifest.json")) if (DATA_DIR / "manifest.json").exists() else {}
print("manifest quota_met:", man.get("quota_met"))

## 4. Load the GHAST dataset & preprocess for DDP
Replaces both `load_dataset("LocalLLaMA/typed-decisions", "all", ...)` calls (`split="train"` here, `split="test"` in the evaluation section).
`state`, `questions`, `gold` stay JSON-encoded strings; `state` decodes to a plain string, which Laya passes through unchanged.
Items are tokenized once and saved to disk so both DDP ranks can read them.

In [ ]:
from datasets import load_dataset, Features, Value

# Explicit schema: `pattern` and `severity` are null on normal_track rows, so don't rely on inference.
GHAST_FEATURES = Features({
    "id": Value("string"), "split": Value("string"), "mmsi": Value("int64"),
    "pattern": Value("string"), "severity": Value("float64"),
    "state": Value("string"), "questions": Value("string"), "gold": Value("string"),
})
GHAST_FILES = {"train": str(TRAIN_JSONL), "test": str(HOLDOUT_JSONL)}   # holdout.jsonl plays the role of "test"

def load_ghast_split(split):
    """Drop-in for load_dataset("LocalLLaMA/typed-decisions", "all", split=split)."""
    return load_dataset("json", data_files=GHAST_FILES, features=GHAST_FEATURES, split=split)

LABELS = ["normal_track", "teleport_jump", "gradual_drift", "freeze_replay", "impossible_kinematics"]

def validate_split(ds, name, per_class):
    from collections import Counter
    counts = Counter()
    assert len(set(ds["id"])) == len(ds), f"{name}: duplicate ids"
    for row in ds:
        assert isinstance(json.loads(row["state"]), str)
        qs, gold = json.loads(row["questions"]), json.loads(row["gold"])
        assert list(qs) == ["pattern"] and qs["pattern"]["type"] == "choice" and list(qs["pattern"]["criteria"]) == LABELS
        counts[gold["pattern"]["label"]] += 1
    assert all(counts[l] == per_class for l in LABELS), f"{name}: unbalanced {dict(counts)}"
    print(f"{name}: {len(ds)} rows OK, per-class = {dict(counts)}")

print("Loading GHAST train split from", TRAIN_JSONL)
ds_train = load_ghast_split("train")
validate_split(ds_train, "train", 1500)
validate_split(load_ghast_split("test"), "holdout", 300)

In [ ]:
import os, json, torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]

    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))

    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[t],
        "target": target,
        "label": label
    }

items, dropped = [], 0
for row in ds_train:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, q, gold[qid])
            if it:
                items.append(it)
            else:
                dropped += 1

lens = [len(i["ids"]) for i in items]
print(f"Preprocessed {len(items)} training sequences across {len(ds_train)} cases (dropped {dropped}).")
print(f"Sequence length tokens: mean {sum(lens)/len(lens):.0f}, max {max(lens)}")
assert dropped == 0, "Some items lost option markers to truncation - check max_len/head_max_len"
torch.save(items, ITEMS_PATH)
print("Saved preprocessed items to", ITEMS_PATH)

## 5. DDP Training Script (`train_ddp.py`)
The original Laya RLCD training script. Only edits: items path from `argv[3]`, epochs from the `LAYA_EPOCHS` env var (default 4), model named `laya-ghast-pattern`, and the saved config's `training` block now describes *this* run instead of inheriting the base checkpoint's numbers.

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    all_items = torch.load(sys.argv[3], weights_only=False)

    # Hold the calibration slice out of training before sharding. Temperatures fitted on
    # items the run has already trained on measure the fit rather than the calibration: the
    # model is near-certain and near-correct on them, so the optimiser has nothing to soften
    # and returns a degenerate scale. The seed is fixed and rank-independent, so every rank
    # withholds exactly the same items and none of them reaches a training batch.
    CALIB_MAX = 400
    order = list(range(len(all_items)))
    random.Random(20260922).shuffle(order)
    n_calib = min(CALIB_MAX, len(all_items) // 10)
    calib_items = [all_items[i] for i in sorted(order[:n_calib])]
    train_items = [all_items[i] for i in sorted(order[n_calib:])]
    # Trim before sharding so every rank gets the same count (drops at most
    # world_size-1 items). Otherwise ranks can run different numbers of
    # micro-batches and DDP deadlocks on the unmatched all-reduce.
    # See https://github.com/NandhaKishorM/laya/issues/678
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]
    
    EPOCHS = int(os.environ.get("LAYA_EPOCHS", "4"))
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        # calib_items was held out above and never entered a training batch
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["training"] = {"dataset": "GHAST data/laya", "epochs": EPOCHS, "world_size": world_size, "train_items": len(train_items), "calib_items": len(calib_items), "hours": round((time.time() - t0) / 3600, 2)}
        cfg["model_name"] = "laya-ghast-pattern"
        cfg["temperature"] = fitted_temps
        # This fit is per type; inherited bucket overrides would hide the new values.
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

## 6. Launch Multi-GPU Fine-Tuning with `torchrun`
Runs on both T4 GPUs in parallel. A rolling `checkpoint_latest/` is written after every epoch.

In [ ]:
EPOCHS = 4          # the first run used 4 (~26 min on 2xT4)
MODEL_DIR = model_dir
cmd = f"LAYA_EPOCHS={EPOCHS} torchrun --standalone --nproc_per_node=2 {SCRIPT_PATH} {MODEL_DIR} {OUTPUT_DIR} {ITEMS_PATH}"
print("Executing DDP training:", cmd)
!{cmd}

## 7. Evaluate on the GHAST holdout (`holdout.jsonl`, 1,500 windows, vessel-disjoint from train)

In [ ]:
import time, json
import numpy as np
import pandas as pd
import laya
from laya.common import ece_score

print("Loading holdout split for evaluation...")
ds_test = load_ghast_split("test")

agent_ft = laya.Agent(str(OUTPUT_DIR), device="cuda")

predictions = []
latencies_ms = []

print(f"Evaluating {len(ds_test)} holdout cases on GPU...")
t0_eval = time.time()

for i, row in enumerate(ds_test):
    case_id = row["id"]
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    workflow = gold["pattern"]["label"]      # GHAST has no "workflow" field; the gold class is the grouping key

    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)

    predictions.append({
        "id": case_id,
        "workflow": workflow,
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })

print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s!")

## 8. Metrics

In [ ]:
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
within_one = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]

    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]

        # 1. Choice
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = str(g_ans["label"])

            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)

            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()

            all_confs.append(float(p_probs.max()))
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))

        # 2. Noul
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = str(g_ans["label"]).lower()

            pred_label = "true" if p_val >= 0.5 else "false"
            is_corr = float(pred_label == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(float(max(p_val, 1.0 - p_val)))

            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])

            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))

        # 3. Score
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            within_one.append(float(abs(p_score - g_score) <= 1.0))

            # Use argmax of probabilities for discrete prediction
            n_levels = len(qdef.get("criteria", []))
            p_probs_score = np.array([p_ans["probabilities"].get(str(i), 0.0) for i in range(n_levels)])
            if p_probs_score.sum() > 0:
                p_probs_score /= p_probs_score.sum()
                p_lvl = int(np.argmax(p_probs_score))
                all_confs.append(float(p_probs_score.max()))
            else:
                p_lvl = int(round(p_score))
                all_confs.append(0.5)

            g_lvl = int(g_ans.get("label", int(round(g_score))))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_mae = np.mean(score_maes) if score_maes else 0.0
laya_within1 = np.mean(within_one) if within_one else 0.0
laya_latency = np.percentile(latencies_ms, 50)

print(f"Accuracy {laya_acc:.3f} | Soft acc {laya_soft_acc:.3f} | Brier {laya_brier:.3f} | ECE {laya_ece:.3f} | p50 latency {laya_latency:.1f} ms")
print("Chance for 5 balanced classes = 0.200")

In [ ]:
# GHAST-specific report: per-class accuracy, confusion matrix, false-positive rate, accuracy by confidence
y_true = [item["gold"]["pattern"]["label"] for item in predictions]
y_pred = [item["pred"]["pattern"]["choice"] for item in predictions]
conf   = np.array([max(item["pred"]["pattern"]["probabilities"].values()) for item in predictions])
hit    = np.array([t == p for t, p in zip(y_true, y_pred)], dtype=float)

cm = pd.crosstab(pd.Series(y_true, name="gold"), pd.Series(y_pred, name="pred")).reindex(index=LABELS, columns=LABELS, fill_value=0)
print("Confusion matrix (rows = gold, cols = predicted)\n", cm.to_string(), "\n")

per_class = {l: float(cm.loc[l, l] / max(1, cm.loc[l].sum())) for l in LABELS}
print("Per-class accuracy:", {k: round(v, 3) for k, v in per_class.items()})

normal = cm.loc["normal_track"]
fp_rate = float(1 - normal["normal_track"] / normal.sum())        # normal tracks called a spoof
spoof_rows = cm.drop(index="normal_track")
miss_rate = float(spoof_rows["normal_track"].sum() / spoof_rows.values.sum())   # spoofs called normal
print(f"normal_track called a spoof (false-positive rate): {fp_rate:.3f}")
print(f"spoof called normal_track (miss rate)            : {miss_rate:.3f}")

conf_table = []
for thr in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    m = conf >= thr
    conf_table.append({"min_confidence": thr, "coverage": round(float(m.mean()), 3),
                       "accuracy": round(float(hit[m].mean()), 3) if m.any() else None})
print("\nAccuracy when confidence >= threshold (use to set PATTERN_MIN_CONFIDENCE):")
print(pd.DataFrame(conf_table).to_string(index=False))

In [ ]:
# Stationary vs moving windows. A vessel that barely moves gives the injectors nothing to distort,
# so a freeze/impossible-kinematics window on it can look identical to a normal one.
import re
def _f(pat, txt):
    m = re.search(pat, txt); return float(m.group(1)) if m else float("nan")
motion = []
for row in ds_test:
    txt = json.loads(row["state"])
    rep_max = _f(r"reported_speed_knots: mean [-\d.]+, max ([-\d.]+)", txt)
    step_km = _f(r"max_step_km: ([-\d.]+)", txt)
    motion.append("stationary" if (rep_max < 1.0 and step_km < 0.1) else "moving")
mdf = pd.DataFrame({"gold": y_true, "hit": hit, "motion": motion})
print("Share of holdout windows that look stationary:", round((mdf.motion == "stationary").mean(), 3))
print(mdf.groupby("motion").hit.agg(["mean", "size"]).round(3).rename(columns={"mean": "accuracy", "size": "n"}).to_string(), "\n")
tab = mdf.pivot_table(index="gold", columns="motion", values="hit", aggfunc="mean").reindex(LABELS).round(3)
print("Accuracy by class and motion:\n", tab.to_string())
motion_table = {"share_stationary": round(float((mdf.motion == "stationary").mean()), 4),
                "accuracy": mdf.groupby("motion").hit.mean().round(4).to_dict(),
                "accuracy_by_class": tab.to_dict()}

## 9. Save report and export the checkpoint
Writes the report, then bundles the files GHAST needs (`model.safetensors`, `rl_agent_config.json`, `encoder/`, `tokenizer/`) into `/kaggle/working/ghast_laya_model/` and a zip. **No Hugging Face push.**

In [ ]:
report = {
    "dataset": "GHAST data/laya (holdout.jsonl)",
    "model": "Laya fine-tuned (Kaggle 2xT4)",
    "n_cases": len(ds_test),
    "chance_accuracy": 0.2,
    "metrics": {
        "accuracy": round(float(laya_acc), 4),
        "soft_accuracy": round(float(laya_soft_acc), 4),
        "brier_score": round(float(laya_brier), 4),
        "ece": round(float(laya_ece), 4),
        "kl_divergence": round(float(laya_kl), 4),
        "total_variation": round(float(laya_tv), 4),
        "latency_p50_ms": round(float(laya_latency), 1),
        "latency_p95_ms": round(float(np.percentile(latencies_ms, 95)), 1),
        "normal_track_false_positive_rate": round(fp_rate, 4),
        "spoof_miss_rate": round(miss_rate, 4),
    },
    "per_class_accuracy": {k: round(v, 4) for k, v in per_class.items()},
    "confusion_matrix": cm.to_dict(),
    "accuracy_by_confidence": conf_table,
    "accuracy_by_motion": motion_table,
}
for path in (WORK_DIR / "laya_ghast_holdout_report.json", OUTPUT_DIR / "benchmark_report.json"):
    path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Saved", path)

# ---- export: only what the scorer needs (see docs/laya_pattern_classifier.md step 4) ----
EXPORT = True
if EXPORT:
    dest = WORK_DIR / "ghast_laya_model"
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    for name in ("model.safetensors", "rl_agent_config.json", "benchmark_report.json"):
        shutil.copy2(OUTPUT_DIR / name, dest / name)
    for name in ("encoder", "tokenizer"):
        shutil.copytree(OUTPUT_DIR / name, dest / name)
    zip_path = shutil.make_archive(str(WORK_DIR / "ghast_laya_model"), "zip", root_dir=dest)
    print(f"Exported checkpoint to {dest}")
    print(f"Zip: {zip_path} ({os.path.getsize(zip_path)/1e6:.0f} MB)")
    print("Download the zip from the notebook's Output tab (after Save Version), unzip it into GHAST/ml/laya_model/,")
    print("then set WITH_LAYA=1 and GHAST_LAYA_MODEL=/laya_model as described in docs/laya_pattern_classifier.md.")